# Google Cloud Dataflow Streaming Pipelines Demonstration (GCS Sinks)

This self-contained notebook demonstrates two end-to-end **Apache Beam (Python)** streaming pipelines running on **Google Cloud Dataflow**:

1. **Pipeline 1: Pub/Sub $\rightarrow$ Dual Cloud Storage (GCS) Destinations (Branching Pipeline)**
   - Reads streaming messages once from a **Cloud Pub/Sub** topic (`beam.io.ReadFromPubSub`).
   - Branches the decoded `PCollection` into **two parallel paths** writing to **two different GCS destinations**:
     - **Branch A (Raw Archive Destination)**: Windows the unmodified messages into `FixedWindows` and writes JSONL files to `gs://<bucket>/pipeline1/raw_archive`.
     - **Branch B (Curated / Enriched Destination)**: Enriches the JSON payload with ingestion metadata (`ingested_at`, `pipeline_branch`), windows into `FixedWindows`, and writes JSONL files to `gs://<bucket>/pipeline1/curated_events`.

2. **Pipeline 2: Continuous GCS Folder Monitoring $\rightarrow$ JSON Mapping $\rightarrow$ Destination GCS Folder**
   - Continuously polls an input **GCS folder** (`gs://<bucket>/incoming/*.json`) for new files using `apache_beam.io.fileio.MatchContinuously` (with stateful file deduplication).
   - Reads each newly arrived file, parses JSON/JSONL payloads, applies a **JSON field mapping and enrichment transformation**, and routes any malformed records to a Dead-Letter Queue (DLQ) branch.
   - Writes the transformed JSON records into a **different GCS folder** (`gs://<bucket>/transformed/`).

---

### Notebook Structure
Each pipeline section is completely self-contained and includes:
- **User Parameters**: Interactive Colab form parameters (`PROJECT_ID`, `REGION`, `BUCKET_NAME`, `PUBSUB_TOPIC_NAME`, etc.).
- **Infrastructure Setup**: Ready-to-execute `gcloud` commands to create the GCS bucket and Pub/Sub topic.
- **Self-Contained Pipeline Code**: Standalone Python scripts written via `%%writefile` so Dataflow workers have clean module state.
- **Dataflow Job Launch**: Ready-to-execute commands to submit the streaming jobs to `DataflowRunner`.
- **Test Payload Generator**: Commands to publish sample events to Pub/Sub and upload sample JSON files to GCS to test the live pipelines.

## 0. Environment Setup & Authentication

Run the cell below to install the Apache Beam GCP SDK (`apache-beam[gcp]`) and authenticate your Colab environment with Google Cloud.

In [ ]:
# Install Apache Beam with GCP support
!pip install --quiet "apache-beam[gcp]>=2.54.0"

# Authenticate with Google Cloud if running in Google Colab
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Cloud via Colab.")
else:
    print("Not running in external Google Colab; using existing environment credentials / ADC.")

---
## 1. Pipeline 1: Read from Pub/Sub and Branch to 2 Different GCS Destinations

### Architecture
```
                          ┌──> [Branch A: Raw Archive] ──────> WindowInto(FixedWindows) ──> fileio.WriteToFiles (GCS Destination 1)
                          │
Pub/Sub Topic ──> ReadFromPubSub ──> Decode UTF-8
                          │
                          └──> [Branch B: Enriched/Curated] ─> Map(enrich_message) ──> WindowInto(FixedWindows) ──> fileio.WriteToFiles (GCS Destination 2)
```

### Key Implementation Details
1. **Single Read, Dual GCS Sink Branching**: `ReadFromPubSub` reads from `projects/<PROJECT_ID>/topics/<PUBSUB_TOPIC_NAME>` once into a `raw_messages` `PCollection`, which is then consumed by two independent branches in the same pipeline:
   - **Branch A (`--output_gcs_path_1`)**: Writes the raw UTF-8 messages directly to the first GCS folder (`pipeline1/raw_archive`).
   - **Branch B (`--output_gcs_path_2`)**: Parses and enriches each event (`enrich_message`) before writing to the second GCS folder (`pipeline1/curated_events`).
2. **Windowed GCS Sinks**: Because Pub/Sub produces an **unbounded** `PCollection`, each branch applies `beam.WindowInto(window.FixedWindows(...))` before calling `apache_beam.io.fileio.WriteToFiles`.

In [ ]:
# @title 1.1 Configure Pipeline 1 Parameters { run: "auto" }

PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
REGION = "us-central1"  # @param {type:"string"}
BUCKET_NAME = "your-gcp-project-id-dataflow-demo"  # @param {type:"string"}
PUBSUB_TOPIC_NAME = "events-demo-topic"  # @param {type:"string"}

# Two Different GCS Destination Folders for Branch A and Branch B
GCS_DESTINATION_1_FOLDER_P1 = "pipeline1/raw_archive"  # @param {type:"string"}
GCS_DESTINATION_2_FOLDER_P1 = "pipeline1/curated_events"  # @param {type:"string"}

# Windowing & GCS Paths for Pipeline 1
WINDOW_SIZE_SECONDS_P1 = 60  # @param {type:"integer"}
GCS_OUTPUT_PATH_1_P1 = f"gs://{BUCKET_NAME}/{GCS_DESTINATION_1_FOLDER_P1}"
GCS_OUTPUT_PATH_2_P1 = f"gs://{BUCKET_NAME}/{GCS_DESTINATION_2_FOLDER_P1}"
TEMP_LOCATION_P1 = f"gs://{BUCKET_NAME}/pipeline1/temp"
STAGING_LOCATION_P1 = f"gs://{BUCKET_NAME}/pipeline1/staging"
PUBSUB_TOPIC_PATH = f"projects/{PROJECT_ID}/topics/{PUBSUB_TOPIC_NAME}"
JOB_NAME_P1 = "pubsub-to-dual-gcs-demo"

print(f"Project ID              : {PROJECT_ID}")
print(f"Region                  : {REGION}")
print(f"Pub/Sub Topic           : {PUBSUB_TOPIC_PATH}")
print(f"GCS Destination 1 (Raw) : {GCS_OUTPUT_PATH_1_P1}")
print(f"GCS Destination 2 (Cur) : {GCS_OUTPUT_PATH_2_P1}")

### 1.2 Create the GCS Bucket and Pub/Sub Topic
Run the cell below to enable the required Google Cloud APIs, create the GCS bucket (`gs://<BUCKET_NAME>`), and create the Pub/Sub topic (`projects/<PROJECT_ID>/topics/<PUBSUB_TOPIC_NAME>`).

In [ ]:
# Set active GCP project
!gcloud config set project "{PROJECT_ID}"

# Enable required GCP APIs (Dataflow, Pub/Sub, Cloud Storage)
!gcloud services enable dataflow.googleapis.com pubsub.googleapis.com storage.googleapis.com --project="{PROJECT_ID}"

# 1. Command to create the GCS bucket (skips if it already exists)
!gcloud storage buckets describe "gs://{BUCKET_NAME}" --project="{PROJECT_ID}" >/dev/null 2>&1 || \
  gcloud storage buckets create "gs://{BUCKET_NAME}" \
    --project="{PROJECT_ID}" \
    --location="{REGION}" \
    --uniform-bucket-level-access

# 2. Command to create the Pub/Sub topic (skips if it already exists)
!gcloud pubsub topics describe "{PUBSUB_TOPIC_NAME}" --project="{PROJECT_ID}" >/dev/null 2>&1 || \
  gcloud pubsub topics create "{PUBSUB_TOPIC_NAME}" \
    --project="{PROJECT_ID}"

### 1.3 Write the Self-Contained Pipeline 1 Script (`pubsub_to_gcs.py`)
Run the cell below to write the complete, self-contained Apache Beam pipeline script to `pubsub_to_gcs.py`.

> **Note on Streaming Job Submission**: Instead of blocking the Colab notebook cell indefinitely with `wait_until_finish()`, this script calls `result = p.run()` and returns as soon as the streaming job is submitted to Dataflow (unless `--wait_until_finish` is explicitly passed).

In [ ]:
%%writefile pubsub_to_gcs.py
"""Self-contained Dataflow Streaming Pipeline 1: Pub/Sub -> 2 Different GCS Destinations."""

import argparse
import datetime
import json
import logging

import apache_beam as beam
from apache_beam import window
from apache_beam.io import fileio
from apache_beam.options.pipeline_options import PipelineOptions
from apache_beam.options.pipeline_options import SetupOptions
from apache_beam.options.pipeline_options import StandardOptions


def enrich_message(raw_msg: str) -> str:
  """Parses a raw JSON string and adds curation metadata for the second GCS branch."""
  now_iso = (
      datetime.datetime.now(datetime.timezone.utc)
      .isoformat()
      .replace("+00:00", "Z")
  )
  try:
    payload = json.loads(raw_msg)
    if not isinstance(payload, dict):
      payload = {"raw_value": payload}
  except json.JSONDecodeError:
    payload = {"raw_text": raw_msg}

  payload["ingested_at"] = now_iso
  payload["pipeline_branch"] = "curated_events"
  return json.dumps(payload)


def run(argv=None):
  parser = argparse.ArgumentParser(
      description="Streaming Dataflow pipeline: Read from Pub/Sub and branch to 2 GCS destinations."
  )
  parser.add_argument(
      "--input_topic",
      required=True,
      help="Input Pub/Sub topic (projects/<project>/topics/<topic>).",
  )
  parser.add_argument(
      "--output_gcs_path_1",
      required=True,
      help="First GCS destination directory for raw archive files (gs://<bucket>/<prefix1>).",
  )
  parser.add_argument(
      "--output_gcs_path_2",
      required=True,
      help="Second GCS destination directory for enriched/curated files (gs://<bucket>/<prefix2>).",
  )
  parser.add_argument(
      "--window_size_sec",
      type=int,
      default=60,
      help="Fixed window duration in seconds for GCS file writes.",
  )
  parser.add_argument(
      "--wait_until_finish",
      action="store_true",
      help="If set, blocks until pipeline termination.",
  )

  known_args, pipeline_args = parser.parse_known_args(argv)

  pipeline_options = PipelineOptions(pipeline_args)
  pipeline_options.view_as(StandardOptions).streaming = True
  pipeline_options.view_as(SetupOptions).save_main_session = True

  p = beam.Pipeline(options=pipeline_options)

  # 1. Read unbounded stream from Pub/Sub once and decode bytes to UTF-8 strings
  raw_messages = (
      p
      | "ReadFromPubSub" >> beam.io.ReadFromPubSub(topic=known_args.input_topic)
      | "DecodeUTF8"
      >> beam.Map(lambda msg: msg.decode("utf-8", errors="replace"))
  )

  # 2a. Branch A: Write unmodified raw messages to GCS Destination 1
  _ = (
      raw_messages
      | "WindowBranch1"
      >> beam.WindowInto(window.FixedWindows(known_args.window_size_sec))
      | "WriteToGCSDestination1"
      >> fileio.WriteToFiles(
          path=known_args.output_gcs_path_1,
          file_naming=fileio.default_file_naming(
              prefix="raw-events", suffix=".jsonl"
          ),
          shards=1,
      )
  )

  # 2b. Branch B: Enrich messages and write to GCS Destination 2
  _ = (
      raw_messages
      | "EnrichForBranch2" >> beam.Map(enrich_message)
      | "WindowBranch2"
      >> beam.WindowInto(window.FixedWindows(known_args.window_size_sec))
      | "WriteToGCSDestination2"
      >> fileio.WriteToFiles(
          path=known_args.output_gcs_path_2,
          file_naming=fileio.default_file_naming(
              prefix="curated-events", suffix=".jsonl"
          ),
          shards=1,
      )
  )

  result = p.run()
  logging.info("Pipeline 1 submitted successfully! Initial state: %s", result.state)
  if known_args.wait_until_finish:
    result.wait_until_finish()


if __name__ == "__main__":
  logging.getLogger().setLevel(logging.INFO)
  run()


### 1.4 Launch Pipeline 1 on Google Cloud Dataflow
Run the cell below to submit `pubsub_to_gcs.py` to **Google Cloud Dataflow** (`DataflowRunner`).

In [ ]:
# Launch Pipeline 1 on Dataflow with both GCS destinations
!python3 pubsub_to_gcs.py \
  --runner=DataflowRunner \
  --project="{PROJECT_ID}" \
  --region="{REGION}" \
  --job_name="{JOB_NAME_P1}" \
  --temp_location="{TEMP_LOCATION_P1}" \
  --staging_location="{STAGING_LOCATION_P1}" \
  --input_topic="{PUBSUB_TOPIC_PATH}" \
  --output_gcs_path_1="{GCS_OUTPUT_PATH_1_P1}" \
  --output_gcs_path_2="{GCS_OUTPUT_PATH_2_P1}" \
  --window_size_sec={WINDOW_SIZE_SECONDS_P1}

print(f"\nView job in Cloud Console: https://console.cloud.google.com/dataflow/jobs/{REGION}?project={PROJECT_ID}")

### 1.5 Send Example Messages to the Pub/Sub Topic & Verify GCS Output
Once the Dataflow workers have started, run the cell below to publish sample JSON event payloads to the Pub/Sub topic and check the output files in GCS.

In [ ]:
import datetime
import json
import subprocess

# Generate 3 example event payloads and publish them to Pub/Sub via gcloud
sample_events = [
    {
        "event_id": "evt-001",
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
        "service": "storage.googleapis.com",
        "method": "storage.buckets.get",
        "principal": "alice@example.com",
        "caller_ip": "203.0.113.10",
    },
    {
        "event_id": "evt-002",
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
        "service": "iam.googleapis.com",
        "method": "google.iam.admin.v1.SetIamPolicy",
        "principal": "bob@example.com",
        "caller_ip": "198.51.100.42",
    },
    {
        "event_id": "evt-003",
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
        "service": "compute.googleapis.com",
        "method": "v1.compute.firewalls.insert",
        "principal": "charlie@example.com",
        "caller_ip": "192.0.2.99",
    },
]

for idx, event in enumerate(sample_events, start=1):
    payload_str = json.dumps(event)
    print(f"Publishing sample event #{idx} to {PUBSUB_TOPIC_NAME}...")
    subprocess.run(
        [
            "gcloud",
            "pubsub",
            "topics",
            "publish",
            PUBSUB_TOPIC_NAME,
            f"--project={PROJECT_ID}",
            f"--message={payload_str}",
        ],
        check=True,
    )

print("\nListing GCS Destination 1 (Raw Archive) files (allow 1-2 window intervals after worker startup):")
!gcloud storage ls "{GCS_OUTPUT_PATH_1_P1}/" || echo "No files in Destination 1 yet (wait for window to close)."

print("\nListing GCS Destination 2 (Curated Events) files (allow 1-2 window intervals after worker startup):")
!gcloud storage ls "{GCS_OUTPUT_PATH_2_P1}/" || echo "No files in Destination 2 yet (wait for window to close)."

---
## 2. Pipeline 2: Continuously Monitor a GCS Folder, Transform JSON Payloads & Write to Another GCS Folder

### Architecture
```
gs://<bucket>/incoming/*.json
       │
       ▼
fileio.MatchContinuously(interval, has_deduplication=True)
       │
       ▼
fileio.ReadMatches() ──> ParDo(ParseAndTransformJsonDoFn)
                                │
                                ├── [main: "transformed"] ──> WindowInto(FixedWindows) ──> fileio.WriteToFiles (gs://<bucket>/transformed/)
                                │
                                └── [side: "dlq"]         ──> WindowInto(FixedWindows) ──> fileio.WriteToFiles (gs://<bucket>/dlq/)
```

### Key Implementation Details
1. **Continuous Folder Monitoring (`fileio.MatchContinuously`)**:
   - Polls `gs://<bucket>/incoming/*.json*` every `POLL_INTERVAL_SECONDS_P2` seconds.
   - `has_deduplication=True` uses Beam stateful processing (`Watch`) to track already-processed file paths so each new file dropped into the folder is processed **only once**.
   - Setting `start_timestamp=Timestamp(0)` ensures files uploaded just before or during Dataflow worker startup are also picked up on the initial poll.
   - *Production Scale Note*: For very high-volume buckets (millions of files), Cloud Storage Pub/Sub Notifications (`gsutil notification create`) paired with `ReadFromPubSub` + `fileio.ReadAllFromText` is recommended to avoid unbounded deduplication state growth.
2. **Flexible JSON Parsing & Mapping (`ParseAndTransformJsonDoFn`)**:
   - Supports single JSON objects, JSON arrays (`[...]`), and newline-delimited JSON Lines (`.jsonl`).
   - Applies a structured JSON mapping (normalizing `event_id`, `user`, `src_ip`, `action`, `severity`, and enriching with `source_file` and `transformed_at` metadata).
   - Routes malformed records to a Dead-Letter Queue (`dlq`) GCS folder.

In [ ]:
# @title 2.1 Configure Pipeline 2 Parameters { run: "auto" }

PROJECT_ID_P2 = "your-gcp-project-id"  # @param {type:"string"}
REGION_P2 = "us-central1"  # @param {type:"string"}
BUCKET_NAME_P2 = "your-gcp-project-id-dataflow-demo"  # @param {type:"string"}

# Monitored Input Folder Pattern & Destination GCS Folders
INPUT_FOLDER_P2 = "pipeline2/incoming"  # @param {type:"string"}
FILE_GLOB_P2 = "*.json*"  # @param {type:"string"}
OUTPUT_FOLDER_P2 = "pipeline2/transformed"  # @param {type:"string"}
DLQ_FOLDER_P2 = "pipeline2/dlq"  # @param {type:"string"}

# Polling & Windowing Intervals
POLL_INTERVAL_SECONDS_P2 = 15  # @param {type:"number"}
WINDOW_SIZE_SECONDS_P2 = 30  # @param {type:"integer"}

INPUT_FILE_PATTERN_P2 = f"gs://{BUCKET_NAME_P2}/{INPUT_FOLDER_P2}/{FILE_GLOB_P2}"
OUTPUT_GCS_PATH_P2 = f"gs://{BUCKET_NAME_P2}/{OUTPUT_FOLDER_P2}"
DLQ_GCS_PATH_P2 = f"gs://{BUCKET_NAME_P2}/{DLQ_FOLDER_P2}"
TEMP_LOCATION_P2 = f"gs://{BUCKET_NAME_P2}/pipeline2/temp"
STAGING_LOCATION_P2 = f"gs://{BUCKET_NAME_P2}/pipeline2/staging"
JOB_NAME_P2 = "gcs-continuous-json-transform-demo"

print(f"Project ID          : {PROJECT_ID_P2}")
print(f"Region              : {REGION_P2}")
print(f"Monitored Pattern   : {INPUT_FILE_PATTERN_P2} (every {POLL_INTERVAL_SECONDS_P2}s)")
print(f"Transformed Output  : {OUTPUT_GCS_PATH_P2}/")
print(f"DLQ Output          : {DLQ_GCS_PATH_P2}/")

### 2.2 Create the GCS Bucket for Pipeline 2
Run the cell below to create the GCS bucket (`gs://<BUCKET_NAME_P2>`) if it does not already exist.

In [ ]:
# Set active GCP project and create the GCS bucket if needed
!gcloud config set project "{PROJECT_ID_P2}"

!gcloud storage buckets describe "gs://{BUCKET_NAME_P2}" --project="{PROJECT_ID_P2}" >/dev/null 2>&1 || \
  gcloud storage buckets create "gs://{BUCKET_NAME_P2}" \
    --project="{PROJECT_ID_P2}" \
    --location="{REGION_P2}" \
    --uniform-bucket-level-access

print(f"Bucket gs://{BUCKET_NAME_P2} is ready.")

### 2.3 Write the Self-Contained Pipeline 2 Script (`gcs_continuous_json_transform.py`)
Run the cell below to write the standalone Apache Beam streaming pipeline script to `gcs_continuous_json_transform.py`.

In [ ]:
%%writefile gcs_continuous_json_transform.py
"""Self-contained Dataflow Streaming Pipeline 2: Continuous GCS Folder Watch -> JSON Mapping -> GCS."""

import argparse
import datetime
import json
import logging
from typing import Any, Iterable

import apache_beam as beam
from apache_beam import pvalue
from apache_beam import window
from apache_beam.io import fileio
from apache_beam.options.pipeline_options import PipelineOptions
from apache_beam.options.pipeline_options import SetupOptions
from apache_beam.options.pipeline_options import StandardOptions
from apache_beam.utils.timestamp import Timestamp


class ParseAndTransformJsonDoFn(beam.DoFn):
  """Reads newly matched GCS files, maps JSON fields to a target schema, and emits JSONL strings."""

  OUTPUT_TAG_DLQ = "dlq"

  def process(
      self, readable_file: fileio.ReadableFile
  ) -> Iterable[Any]:
    file_path = readable_file.metadata.path
    logging.info("Processing newly detected GCS file: %s", file_path)

    try:
      raw_text = readable_file.read_utf8().strip()
    except Exception as exc:  # pylint: disable=broad-except
      yield pvalue.TaggedOutput(
          self.OUTPUT_TAG_DLQ,
          json.dumps({
              "source_file": file_path,
              "error": f"Failed to read UTF-8 file content: {exc}",
          }),
      )
      return

    if not raw_text:
      return

    # Support both a full JSON document (object or list) and JSON Lines (.jsonl)
    records_to_map: list[tuple[int, str, dict[str, Any] | None, str | None]] = []
    try:
      parsed_whole = json.loads(raw_text)
      if isinstance(parsed_whole, list):
        for idx, item in enumerate(parsed_whole, start=1):
          if isinstance(item, dict):
            records_to_map.append((idx, json.dumps(item), item, None))
          else:
            records_to_map.append((
                idx,
                json.dumps(item),
                None,
                "Expected JSON object element inside array",
            ))
      elif isinstance(parsed_whole, dict):
        records_to_map.append((1, raw_text, parsed_whole, None))
      else:
        records_to_map.append(
            (1, raw_text, None, "Top-level JSON must be an object or array")
        )
    except json.JSONDecodeError:
      # Fallback: parse line-by-line as JSON Lines (JSONL)
      for line_num, line in enumerate(raw_text.splitlines(), start=1):
        stripped = line.strip()
        if not stripped:
          continue
        try:
          obj = json.loads(stripped)
          if isinstance(obj, dict):
            records_to_map.append((line_num, stripped, obj, None))
          else:
            records_to_map.append(
                (line_num, stripped, None, "JSONL line is not a JSON object")
            )
        except json.JSONDecodeError as err:
          records_to_map.append((line_num, stripped, None, str(err)))

    now_iso = (
        datetime.datetime.now(datetime.timezone.utc)
        .isoformat()
        .replace("+00:00", "Z")
    )

    for record_idx, raw_record, record, error_msg in records_to_map:
      if error_msg or record is None:
        yield pvalue.TaggedOutput(
            self.OUTPUT_TAG_DLQ,
            json.dumps({
                "source_file": file_path,
                "record_index": record_idx,
                "raw_payload": raw_record,
                "error": error_msg,
                "failed_at": now_iso,
            }),
        )
        continue

      # --- Example JSON Mapping / Normalization ---
      mapped_Keys = {
          "event_id",
          "timestamp",
          "user",
          "src_ip",
          "dest_ip",
          "action",
          "severity",
          "message",
      }
      transformed_payload = {
          "id": str(record.get("event_id", f"evt-{record_idx}")),
          "event_timestamp": record.get("timestamp", now_iso),
          "principal": {
              "user_id": record.get("user", "unknown"),
              "ip": record.get("src_ip", "0.0.0.0"),
          },
          "target": {
              "ip": record.get("dest_ip", "0.0.0.0"),
          },
          "security_result": {
              "action": str(record.get("action", "UNKNOWN")).upper(),
              "severity": str(record.get("severity", "INFO")).upper(),
              "description": record.get("message", ""),
          },
          "additional_fields": {
              k: v for k, v in record.items() if k not in mapped_Keys
          },
          "ingestion_metadata": {
              "source_file": file_path,
              "record_index": record_idx,
              "transformed_at": now_iso,
          },
      }
      yield json.dumps(transformed_payload)


def run(argv=None):
  parser = argparse.ArgumentParser(
      description="Streaming Dataflow pipeline: Continuous GCS folder monitor & JSON mapper."
  )
  parser.add_argument(
      "--input_file_pattern",
      required=True,
      help="GCS file glob pattern to monitor (e.g. gs://<bucket>/incoming/*.json*).",
  )
  parser.add_argument(
      "--output_gcs_path",
      required=True,
      help="Destination GCS folder for transformed JSONL files.",
  )
  parser.add_argument(
      "--dlq_gcs_path",
      required=True,
      help="Destination GCS folder for Dead-Letter Queue (malformed) files.",
  )
  parser.add_argument(
      "--poll_interval_sec",
      type=float,
      default=15.0,
      help="Polling interval in seconds for MatchContinuously.",
  )
  parser.add_argument(
      "--window_size_sec",
      type=int,
      default=30,
      help="Fixed window duration in seconds for writing output files to GCS.",
  )
  parser.add_argument(
      "--wait_until_finish",
      action="store_true",
      help="If set, blocks until pipeline termination.",
  )

  known_args, pipeline_args = parser.parse_known_args(argv)

  pipeline_options = PipelineOptions(pipeline_args)
  pipeline_options.view_as(StandardOptions).streaming = True
  pipeline_options.view_as(SetupOptions).save_main_session = True

  p = beam.Pipeline(options=pipeline_options)

  # 1. Continuously monitor the GCS folder pattern and read matched files
  results = (
      p
      | "MatchFilesContinuously"
      >> fileio.MatchContinuously(
          file_pattern=known_args.input_file_pattern,
          interval=known_args.poll_interval_sec,
          has_deduplication=True,
          start_timestamp=Timestamp(0),
      )
      | "ReadMatchedFiles" >> fileio.ReadMatches()
      | "ParseAndTransformJSON"
      >> beam.ParDo(ParseAndTransformJsonDoFn()).with_outputs(
          ParseAndTransformJsonDoFn.OUTPUT_TAG_DLQ,
          main="transformed",
      )
  )

  # 2a. Write transformed JSONL records to the destination GCS folder
  _ = (
      results.transformed
      | "WindowTransformedRecords"
      >> beam.WindowInto(window.FixedWindows(known_args.window_size_sec))
      | "WriteTransformedToGCS"
      >> fileio.WriteToFiles(
          path=known_args.output_gcs_path,
          file_naming=fileio.default_file_naming(
              prefix="transformed-events", suffix=".jsonl"
          ),
          shards=1,
      )
  )

  # 2b. Write any malformed records to the DLQ GCS folder
  _ = (
      results[ParseAndTransformJsonDoFn.OUTPUT_TAG_DLQ]
      | "WindowDLQRecords"
      >> beam.WindowInto(window.FixedWindows(known_args.window_size_sec))
      | "WriteDLQToGCS"
      >> fileio.WriteToFiles(
          path=known_args.dlq_gcs_path,
          file_naming=fileio.default_file_naming(
              prefix="dlq-records", suffix=".jsonl"
          ),
          shards=1,
      )
  )

  result = p.run()
  logging.info("Pipeline 2 submitted successfully! Initial state: %s", result.state)
  if known_args.wait_until_finish:
    result.wait_until_finish()


if __name__ == "__main__":
  logging.getLogger().setLevel(logging.INFO)
  run()


### 2.4 Launch Pipeline 2 on Google Cloud Dataflow
Run the cell below to submit `gcs_continuous_json_transform.py` to **Google Cloud Dataflow** (`DataflowRunner`).

In [ ]:
# Launch Pipeline 2 on Dataflow
!python3 gcs_continuous_json_transform.py \
  --runner=DataflowRunner \
  --project="{PROJECT_ID_P2}" \
  --region="{REGION_P2}" \
  --job_name="{JOB_NAME_P2}" \
  --temp_location="{TEMP_LOCATION_P2}" \
  --staging_location="{STAGING_LOCATION_P2}" \
  --input_file_pattern="{INPUT_FILE_PATTERN_P2}" \
  --output_gcs_path="{OUTPUT_GCS_PATH_P2}" \
  --dlq_gcs_path="{DLQ_GCS_PATH_P2}" \
  --poll_interval_sec={POLL_INTERVAL_SECONDS_P2} \
  --window_size_sec={WINDOW_SIZE_SECONDS_P2}

print(f"\nView job in Cloud Console: https://console.cloud.google.com/dataflow/jobs/{REGION_P2}?project={PROJECT_ID_P2}")

### 2.5 Upload an Example JSON Payload to the Monitored GCS Folder & Verify Output
Run the cell below to create a sample JSON file containing security events and copy it into `gs://<BUCKET_NAME_P2>/<INPUT_FOLDER_P2>/`. Pipeline 2 will detect the new file on its next poll interval, transform the JSON records, and write them to `gs://<BUCKET_NAME_P2>/<OUTPUT_FOLDER_P2>/`.

In [ ]:
import datetime
import json
import time

# 1. Create a local example JSON file with sample events
timestamp_suffix = int(time.time())
sample_filename = f"/tmp/sample_events_{timestamp_suffix}.json"
gcs_target_uri = f"gs://{BUCKET_NAME_P2}/{INPUT_FOLDER_P2}/sample_events_{timestamp_suffix}.json"

example_payload = [
    {
        "event_id": "evt-1001",
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
        "user": "jdoe@corp.example.com",
        "src_ip": "10.24.16.8",
        "dest_ip": "172.16.0.10",
        "action": "allow",
        "severity": "low",
        "message": "SSH login succeeded",
        "device_hostname": "bastion-us-central1-a",
        "port": 22,
    },
    {
        "event_id": "evt-1002",
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
        "user": "svc-deploy@corp.example.com",
        "src_ip": "198.51.100.77",
        "dest_ip": "172.16.0.55",
        "action": "block",
        "severity": "high",
        "message": "Unauthorized outbound connection blocked by firewall",
        "protocol": "TCP",
        "port": 4444,
    },
]

with open(sample_filename, "w", encoding="utf-8") as f:
    json.dump(example_payload, f, indent=2)

# 2. Upload the sample JSON file to the monitored GCS folder
print(f"Uploading {sample_filename} -> {gcs_target_uri} ...")
!gcloud storage cp "{sample_filename}" "{gcs_target_uri}" --project="{PROJECT_ID_P2}"

print("\nMonitored input folder contents:")
!gcloud storage ls "gs://{BUCKET_NAME_P2}/{INPUT_FOLDER_P2}/"

print("\nTransformed output folder contents (wait ~1-2 window intervals after worker startup):")
!gcloud storage ls "{OUTPUT_GCS_PATH_P2}/" || echo "No transformed files written yet (wait for window to close)."

---
## 3. Inspect Output Files & Clean Up Running Streaming Jobs

Run the cell below to view the contents of the output files in GCS or to list and cancel active streaming Dataflow jobs when you have finished testing.

In [ ]:
# Inspect output from Pipeline 1 (both GCS branches) and Pipeline 2 (once windows have closed)
print("=== Pipeline 1 — Branch A: Raw Archive Files Content (GCS Destination 1) ===")
!gcloud storage cat "{GCS_OUTPUT_PATH_1_P1}/*" 2>/dev/null | head -n 20 || echo "No Pipeline 1 Branch A output files available yet."

print("\n=== Pipeline 1 — Branch B: Curated Events Files Content (GCS Destination 2) ===")
!gcloud storage cat "{GCS_OUTPUT_PATH_2_P1}/*" 2>/dev/null | head -n 20 || echo "No Pipeline 1 Branch B output files available yet."

print("\n=== Pipeline 2 — Transformed Output Files Content (GCS -> JSON Mapping -> GCS) ===")
!gcloud storage cat "{OUTPUT_GCS_PATH_P2}/*" 2>/dev/null | head -n 20 || echo "No Pipeline 2 transformed files available yet."

# List active Dataflow jobs in the project/region
print("\n=== Active Dataflow Jobs ===")
!gcloud dataflow jobs list --project="{PROJECT_ID}" --region="{REGION}" --status=active

# To cancel a running streaming job, uncomment and run with the JOB_ID from above:
# !gcloud dataflow jobs cancel <JOB_ID> --project="{PROJECT_ID}" --region="{REGION}"